In [1]:
from google.colab import drive
drive.mount('/content/drive')





Mounted at /content/drive


In [2]:
!pip install -q ultralytics pandas pyyaml
import os
import cv2
import matplotlib.pyplot as plt
DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/3001"

for folder in ["train", "valid", "test"]:
    path = os.path.join(DATA_ROOT, folder)
    print(folder, os.path.isdir(path), path)

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import yaml
import torch
from ultralytics import YOLO

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/3001")

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"

# Generated dataset and model outputs
YOLO_DATASET = Path("/content/yolo_dataset")
RUNS_DIR = BASE_DIR / "yolo_original_results"

print("GPU available:", torch.cuda.is_available())
print("Training folder exists:", TRAIN_DIR.exists())
print("Test folder exists:", TEST_DIR.exists())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.3 MB/s eta 0:00:00
train True /content/drive/MyDrive/Colab Notebooks/3001/train
valid True /content/drive/MyDrive/Colab Notebooks/3001/valid
test True /content/drive/MyDrive/Colab Notebooks/3001/test
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU available: True
Training folder exists: True
Test folder exists: True


In [5]:
import random

random.seed(42)
np.random.seed(42)

CLASS_NAMES = [
    "loose-meat",
    "twisted-meat",
    "unsealed",
    "wrinkle"
]

# Map CSV class labels to IDs 0–3.
# Adjust aliases below to match the exact labels in your CSV.
CLASS_ALIASES = {
    "loose-meat": 0,
    "loose_meat": 0,
    "loose": 0,
    "twisted-meat": 1,
    "twisted_meat": 1,
    "twisted": 1,
    "unsealed": 2,
    "unsealed packaging": 2,
    "unsealed_packaging": 2,
    "packaging-error": 2,
    "wrinkled": 3,
    "wrinkled_meat": 3,
    "wrinkle": 3
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_csv(folder):
    csvs = list(folder.glob("*.csv"))
    if len(csvs) != 1:
        raise ValueError(
            f"Expected exactly one CSV in {folder}, found: {[p.name for p in csvs]}"
        )
    return csvs[0]

def convert_split(source_dir, split_name, csv_path=None):
    csv_path = csv_path or find_csv(source_dir)
    df = pd.read_csv(csv_path)

    required = {
        "filename", "width", "height", "class",
        "xmin", "ymin", "xmax", "ymax"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing CSV columns in {csv_path.name}: {missing}")

    # Look up image files by filename.
    image_lookup = {
        p.name: p for p in source_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    }

    # Group all bounding boxes belonging to each image.
    grouped = {
        str(name): group
        for name, group in df.groupby("filename")
    }

    # Keep annotated images together when making a validation split.
    filenames = sorted(grouped.keys())
    random.Random(42).shuffle(filenames)

    if split_name == "train":
        if len(filenames) < 2:
            raise ValueError("Need at least two annotated training images.")
        n_val = max(1, round(0.2 * len(filenames)))
        val_names = set(filenames[:n_val])
        train_names = set(filenames[n_val:])
        assignments = [
            ("train", train_names),
            ("val", val_names)
        ]
    else:
        assignments = [(split_name, set(filenames))]

    for target_split, names in assignments:
        img_out = YOLO_DATASET / "images" / target_split
        lbl_out = YOLO_DATASET / "labels" / target_split
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        for filename in names:
            if filename not in image_lookup:
                print(f"WARNING: Image not found, skipping: {filename}")
                continue

            group = grouped[filename]
            src = image_lookup[filename]
            dst = img_out / src.name
            shutil.copy2(src, dst)

            # Write one YOLO label file per image.
            lines = []
            for _, row in group.iterrows():
                raw_class = str(row["class"]).strip().lower()
                if raw_class not in CLASS_ALIASES:
                    raise ValueError(
                        f"Unknown class '{row['class']}' in {csv_path.name}. "
                        "Add it to CLASS_ALIASES."
                    )

                class_id = CLASS_ALIASES[raw_class]
                width = float(row["width"])
                height = float(row["height"])

                xmin = float(row["xmin"])
                ymin = float(row["ymin"])
                xmax = float(row["xmax"])
                ymax = float(row["ymax"])

                if width <= 0 or height <= 0:
                    raise ValueError(f"Invalid image dimensions for {filename}")

                # YOLO uses normalised centre coordinates and box dimensions.
                x_center = ((xmin + xmax) / 2) / width
                y_center = ((ymin + ymax) / 2) / height
                box_width = (xmax - xmin) / width
                box_height = (ymax - ymin) / height

                # Check coordinates are valid.
                if not (0 <= x_center <= 1 and 0 <= y_center <= 1
                        and 0 < box_width <= 1 and 0 < box_height <= 1):
                    raise ValueError(
                        f"Invalid bounding box in {filename}: "
                        f"{xmin, ymin, xmax, ymax}"
                    )

                lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} "
                    f"{box_width:.6f} {box_height:.6f}"
                )

            (lbl_out / f"{src.stem}.txt").write_text(
                "\n".join(lines), encoding="utf-8"
            )

    print(f"Converted annotations from {csv_path}")

# Start clean so old generated labels cannot contaminate this run.
if YOLO_DATASET.exists():
    shutil.rmtree(YOLO_DATASET)

convert_split(TRAIN_DIR, "train")
convert_split(TEST_DIR, "test")

# Write YOLO dataset configuration.
data_config = {
    "path": str(YOLO_DATASET),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)}
}

config_path = YOLO_DATASET / "data.yaml"
with open(config_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("\nDataset configuration:")
print(config_path.read_text())

Converted annotations from /content/drive/MyDrive/Colab Notebooks/3001/train/_annotations.csv
Converted annotations from /content/drive/MyDrive/Colab Notebooks/3001/test/_annotations.csv

Dataset configuration:
path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test
names:
  0: loose-meat
  1: twisted-meat
  2: unsealed
  3: wrinkle



In [7]:
model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(config_path),
    epochs=25,
    imgsz=640,
    batch=16,
    patience=5,
    project=str(RUNS_DIR),
    name="original_images",
    exist_ok=True,
    plots=True,
    seed=42
)

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=original_images, nbs=64, nms=None, opset=None, optimize=Fal

In [8]:
best_model_path = RUNS_DIR / "original_images" / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

test_results = best_model.val(
    data=str(config_path),
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    project=str(RUNS_DIR),
    name="original_test",
    exist_ok=True
)

print("\nTest results")
print(f"mAP50:     {test_results.box.map50 * 100:.2f}%")
print(f"mAP50-95:  {test_results.box.map * 100:.2f}%")
print(f"Precision: {test_results.box.mp * 100:.2f}%")
print(f"Recall:    {test_results.box.mr * 100:.2f}%")

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3868.6±862.2 MB/s, size: 236.6 KB)
val: Scanning /content/yolo_dataset/labels/test... 75 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 75/75 2.0Kit/s 0.0s
val: New cache created: /content/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.7s
                   all         75         91      0.734       0.47        0.6      0.235
            loose-meat          2          3          1      0.663      0.736      0.213
          twisted-meat          7          7      0.363      0.429      0.449        0.2
              unsealed         67         81      0.838       0.32      0.615      0.291
Speed: 4.6ms preprocess, 7.6ms inference, 0.0ms loss, 2.8ms postprocess per imag

In [9]:
prediction_dir = RUNS_DIR / "test_predictions"

best_model.predict(
    source=str(YOLO_DATASET / "images" / "test"),
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(prediction_dir),
    name="predictions",
    exist_ok=True
)

print("Predictions saved in:", prediction_dir / "predictions")


image 1/75 /content/yolo_dataset/images/test/frame_with_red_102_jpg.rf.85bc1050e842e86cc905276833e958f0.jpg: 480x640 1 unsealed, 9.4ms
image 2/75 /content/yolo_dataset/images/test/frame_with_red_121_jpg.rf.79a8b4ab4a2f4b92c456d4cec1174733.jpg: 480x640 2 twisted-meats, 9.5ms
image 3/75 /content/yolo_dataset/images/test/frame_with_red_1271_jpg.rf.e8adabc98808cf059c609afc6e6a4ece.jpg: 480x640 1 unsealed, 10.2ms
image 4/75 /content/yolo_dataset/images/test/frame_with_red_1298_jpg.rf.d222da0198279b3bf8b301470dba21ab.jpg: 480x640 1 unsealed, 10.3ms
image 5/75 /content/yolo_dataset/images/test/frame_with_red_1405_jpg.rf.0d8b7b217638f7d7d447091d7fe6942e.jpg: 480x640 1 unsealed, 10.2ms
image 6/75 /content/yolo_dataset/images/test/frame_with_red_1427_jpg.rf.fd24ce8d23802f763dd4728299fd00e4.jpg: 480x640 1 unsealed, 9.4ms
image 7/75 /content/yolo_dataset/images/test/frame_with_red_1458_jpg.rf.ba5eaf69e5e0b85e8f2a06efd2114f52.jpg: 480x640 (no detections), 9.4ms
image 8/75 /content/yolo_dataset/ima

In [11]:
class_names = [
    "Loose meat",
    "Twisted meat",
    "Unsealed packaging",
    "Wrinkled meat"
]

print("\nPercentage correctly detected for each defect category")
print("-" * 55)

# Map evaluation indices to class IDs using the model's test results mapping
# If a class is completely absent in the test set, its recall is printed as N/A or 0.00%
for i, class_name in enumerate(class_names):
    # test_results.ap_class_index contains the dataset class indices that were evaluated
    if hasattr(test_results, 'ap_class_index') and i in test_results.ap_class_index:
        eval_idx = list(test_results.ap_class_index).index(i)
        recall = test_results.box.r[eval_idx] * 100
        print(f"{class_name}: {recall:.2f}%")
    else:
        print(f"{class_name}: 0.00% (No test instances)")


Percentage correctly detected for each defect category
-------------------------------------------------------
Loose meat: 66.30%
Twisted meat: 42.86%
Unsealed packaging: 31.99%
Wrinkled meat: 0.00% (No test instances)
